<a href="https://colab.research.google.com/github/atikhasan007/Generative-AI/blob/main/LangGraph_ReAct_Mini_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install langchain-core langchain-community langgraph langchain-openai duckduckgo-search


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


# LangGraph ReAct Function Calling Pattern

* Search
* Math


# Traditionall ReAct
react_prompt = '''Assistant is a large language model trained by Microsoft. It can generate human-like text to help with a wide range of tasks, from answering questions to providing detailed explanations and discussions. It processes large amounts of text, continuously learns, and delivers relevant, informative, and coherent responses. Overall, it is a powerful and versatile tool for assistance and insights.'''

# Tools

Assistant has access to the following tools:

* wikipedia_search - searches the    wikipedia database for the answer
* web_search - searches the web for the answer
* calculator - calculates the answer to the question
* weather_api - gets the weather for the location

To use a tool, please use the following format:

#### Thought: Do I need to use a tool? Yes

#### Action: the action to take, should be one of [wikipedia_search, web_search, calculator, weather_api]
#### Action Input: the input to the action
####Observation: the result of the action

## When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:

* Thought: Do I need to use a tool? No
* Final Answer: [your response here]


##### Begin!

New input: Whow was King Arthur? """

In [ ]:
import os
from google.colab import userdata
os.environ['OpenAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model_name='gpt-3.5-turbo',
    temperature=0)

# Tools

In [ ]:
def multiply(a:int, b:int)-> int:
  """Multiply a and b.
  Args:
      a: The first number.
      b: The second number.
      Returns:
      The product of a and b.
  """
  return a*b



# This will be a tool
def add(a:int, b:int)->int:
  """Adds a and b.
  Args:
      a: The first number.
      b: The second number.
      Returns:
      The sum of a and b.
  """
  return a+b


def divide(a:int, b:int)->int:
  """Divides a and b.
  Args:
      a: The first number.
      b: The second number.
      Returns:
      The quotient of a and b.
  """
  return a/b


In [ ]:
# search tools
from langchain_community.tools import DuckDuckGoSearchRun
search = DuckDuckGoSearchRun()

search.invoke("Who was King Arthur?")


In [ ]:
tools = [add, multiply, divide, search]
llm_with_tools = llm.bind(tools)

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

# system message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with using seach and performing arithmetic on a set of inputs.")


# Nodes

In [ ]:
# Node
def reasoner(state: MessagesState):
  return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# Building the graph


In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition # this is the checker for the if you got a tool back
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display
# Graph
builder = StateGraph(MessagesState)
# Add nodes
builder.add_node("reasoner", reasoner)
builder.add_node("tools", ToolNode(tools)) # for the tools
# Add edges
builder.add_edge(START, "reasoner")
builder.add_conditional_edges(
    "reasoner",
    # If the latest message (result) from node reasoner is a tool call -> tools_condition routes to tools
    # If the latest message (result) from node reasoner is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", "reasoner")
react_graph = builder.compile()

# Display the graph
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
messages = [HumanMessage(content="What is 2 times Brad Pitt's age?")]
messages = react_graph.invoke({"messages": messages})

# More manual way and adding a custom tool

In [2]:
!pip -q install yahoo-finance

  Preparing metadata (setup.py) ... done


In [ ]:
import yfinance as yf

def get_stock_price(ticker: str) -> float:
    """Gets a stock price from Yahoo Finance.

    Args:
        ticker: ticker str
    """
    # """This is a tool for getting the price of a stock when passed a ticker symbol"""
    stock = yf.Ticker(ticker)
    return stock.info['previousClose']

In [ ]:
get_stock_price("AAPL")

In [3]:
# Node
def reasoner(state):
    query = state["query"]
    messages = state["messages"]
    # System message
    sys_msg = SystemMessage(content="You are a helpful assistant tasked with using search, the yahoo finance tool and performing arithmetic on a set of inputs.")
    message = HumanMessage(content=query)
    messages.append(message)
    result = [llm_with_tools.invoke([sys_msg] + messages)]
    return {"messages":result}


In [ ]:
tools = [add, multiply, divide, search, get_stock_price]

llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
tools[4]

In [ ]:
from typing import Annotated, TypedDict
import operator
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages


class GraphState(TypedDict):
    """State of the graph."""
    query: str
    finance: str
    final_answer: str
    # intermediate_steps: Annotated[list[tuple[AgentAction, str]], operator.add]
    messages: Annotated[list[AnyMessage], operator.add]


In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition # this is the checker for the
from langgraph.prebuilt import ToolNode


# Graph
workflow = StateGraph(GraphState)

# Add Nodes
workflow.add_node("reasoner", reasoner)
workflow.add_node("tools", ToolNode(tools)) # for the tools

# Add Edges
workflow.add_edge(START, "reasoner")

workflow.add_conditional_edges(
    "reasoner",
    # If the latest message (result) from node reasoner is a tool call -> tools_condition routes to tools
    # If the latest message (result) from node reasoner is a not a tool call -> tools_condition routes to END
    tools_condition,
)
workflow.add_edge("tools", "reasoner")
react_graph = workflow.compile()

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
response = react_graph.invoke({"query": "What is 2 times Brad Pitt's age?", "messages": []})

In [ ]:
response['messages'][-1].pretty_print()

In [ ]:
for m in response['messages']:
    m.pretty_print()

In [ ]:
response = react_graph.invoke({"query": "What is the stock price of the company that Jensen Huang is CEO of?", "messages": []})

In [ ]:
for m in response['messages']:
    m.pretty_print()

In [ ]:
response = react_graph.invoke({"query": "What will be the price of nvidia stock if it doubles?", "messages": []})

In [ ]:
for m in response['messages']:
    m.pretty_print()